# Трек C: C2 (Grounding DINO, OWLv2) и C5 (псевдометки) на Kaggle GPU

Вся логика в скриптах `tracks/ov/`, ноутбук их только запускает. Состояние работы и измерения: `tracks/ov/HANDOFF.md`.

**Перед запуском**

1. Settings: Accelerator `GPU T4 x2`. Internet `On`, пока веса не лежат в подключённом датасете.
2. Add Input: датасет соревнования; датасет с кодом репозитория; при наличии датасет с весами (`models/ov_gdino`, `models/ov_owl`) и вывод прошлого запуска этого ноутбука.
3. В первой ячейке указать `TS_GIT_SHA` (коммит архива с кодом), `PROMPT_VERSION` и `RUN_TRAIN_COMPARE`.

**Порядок**

- Первый запуск интерактивно: ячейки 0–2 (настройка, короткая проверка, сравнение промптов на train). По таблицам выбрать `PROMPT_VERSION`.
- Второй запуск: `RUN_TRAIN_COMPARE = False`, Save Version → Save & Run All. Считаются только целевые кадры (`dt = 0`, 4964 кропа на модель).
- Вывод (`/kaggle/working/artifacts`) сохранить как версию датасета команды. Если прогон не уложился во время, подключить вывод как Input и запустить ещё раз: расчёт продолжится с места остановки.

Тестовые кадры обрабатываются только автоматически. Пороги подбираются только по счётчикам train. Листы для просмотра глазами строятся только по train-строкам.

In [ ]:
import os
from pathlib import Path

os.environ['TS_GIT_SHA'] = ''   # коммит репозитория, из которого собран архив с кодом
PROMPT_VERSION = 1              # промпты tracks/ov/prompts/*_v<P>.txt для основного прогона
RUN_TRAIN_COMPARE = True        # сравнить промпты 1, 2, 3 на train (ячейка 2); для основного прогона False

os.environ['OV_PROMPT'] = str(PROMPT_VERSION)
found = sorted(p.parents[2] for p in Path('/kaggle/input').rglob('tracks/ov/run_ov.py'))
assert len(found) == 1, f'Нужен ровно один подключённый датасет с кодом репозитория, найдено: {found}'
os.chdir(found[0])
%run tracks/ov/kaggle_setup.py

## 1. Короткая проверка

По 8 train-кропов на модель: формат 8.1, скорость и пик памяти GPU. Файлы проверки удаляются. Если OWLv2 падает с `--fp16`, уберите этот флаг здесь и ниже.

In [ ]:
!python tracks/ov/run_ov.py --source ov_gdino --smoke 8
!python tracks/ov/run_ov.py --source ov_owl --smoke 8 --batch 2 --fp16

## 2. Train: сравнение промптов до основного прогона

231 целевая train-строка на модель и версию промпта (отладочные артефакты `v91`–`v93`). Для каждой версии: items и dishes по дням, ИК и без выброса, затем пробный подбор порогов псевдометок (сколько телефонов и тарелок проходит при оценке precision не ниже 0.85).

In [ ]:
if RUN_TRAIN_COMPARE:
    for p in [1, 2, 3]:
        v = 90 + p
        print(f'\n===== промпт v{p} =====')
        !python tracks/ov/run_ov.py --source ov_gdino --rows train0 --version {v} --prompt-version {p}
        !python tracks/ov/run_ov.py --source ov_owl --rows train0 --version {v} --prompt-version {p} --batch 2 --fp16
        !python tracks/ov/eval_dets.py ov_gdino__v{v} ov_owl__v{v}
        !python tracks/ov/make_pseudo.py ov_gdino__v{v} ov_owl__v{v} --version {v}

## 3. Основной прогон: целевые кадры (`dt = 0`)

4964 кропа на модель, сырые детекции с низким порогом пишутся в `artifacts/dets/ov_gdino__v1` и `ov_owl__v1`. Две модели параллельно на двух GPU (на одном GPU по очереди). Прогресс копится в `artifacts/dets/<имя>.parts/`; после `--max-minutes` расчёт останавливается штатно и продолжается при следующем запуске.

In [ ]:
%%bash
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  python tracks/ov/run_ov.py --source ov_gdino --rows targets --prompt-version "$OV_PROMPT" --device cuda:0 --max-minutes 300 &
  python tracks/ov/run_ov.py --source ov_owl --rows targets --prompt-version "$OV_PROMPT" --device cuda:1 --batch 2 --fp16 --max-minutes 300 &
  wait
else
  python tracks/ov/run_ov.py --source ov_gdino --rows targets --prompt-version "$OV_PROMPT" --max-minutes 150
  python tracks/ov/run_ov.py --source ov_owl --rows targets --prompt-version "$OV_PROMPT" --batch 2 --fp16 --max-minutes 150
fi
ls -la "$TS_ARTIFACTS/dets"

## 4. C5 и артефакты для C2

- `artifacts/labels/ov_pseudo__v1.parquet`: согласованные детекции двух моделей, пороги только по счётчикам train.
- `artifacts/rowfeats/ov_counts__v1.parquet`: счётчики items и dishes по всем 4964 строкам, по колонке на вариант. По ним трек A проверяет замену колонок в сабмите.

In [ ]:
art = Path(os.environ['TS_ARTIFACTS'])
ready = all((art / 'dets' / f'{s}__v1.parquet').is_file() for s in ['ov_gdino', 'ov_owl'])
if ready:
    !python tracks/ov/make_pseudo.py ov_gdino__v1 ov_owl__v1
    !python tracks/ov/eval_dets.py ov_gdino__v1 ov_owl__v1 --rowfeats 1
else:
    print('Основной прогон не завершён: сохраните версию, подключите её вывод как Input и запустите ноутбук ещё раз.')

## 5. Оценка стекером трека A (если в архиве код из main)

Числа по контракту даёт `tools/ablate.py`: привязка `ts.assign`, пороги по дням, out-of-fold. Конфиг временный, `configs/` не меняется. Ячейка локально не проверялась: в ветке трека C стекера ещё не было.

In [ ]:
import yaml

if ready and Path('tools/ablate.py').is_file():
    cfg = yaml.safe_load(Path('configs/default.yaml').read_text(encoding='utf-8'))
    cfg['sources']['dets'] = ['prod', 'ov_gdino__v1', 'ov_owl__v1']
    cfg['stack']['counts']['items'] = 'cur'
    tmp = Path('/kaggle/working/ov_sources.yaml')
    tmp.write_text(yaml.safe_dump(cfg, allow_unicode=True), encoding='utf-8')
    !python tools/ablate.py --config {tmp}
else:
    print('Пропуск: нет детекций v1 или в архиве нет tools/ablate.py (код не из main).')

## 6. Что получилось

In [ ]:
from IPython.display import Image, display

for path in sorted(p for kind in ['dets', 'labels', 'rowfeats'] for p in (art / kind).glob('*') if p.is_file()):
    print(f'{path.stat().st_size / 2 ** 20:9.2f} МБ  {path.relative_to(art)}')
for sheet in sorted((art / 'labels').glob('ov_pseudo__v1_train_p*.jpg')):   # только train-кадры
    display(Image(filename=str(sheet), width=1200))